# LeanRank KG Demo

This notebook inspects the processed LeanRank proof knowledge graph outputs and calls the retrieval API. Run the pipeline first:

```bash
leanrank-kg full-pipeline --config configs/leanrank_real_sample.yaml --debug-rows 120
```

In [ ]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
REPORTS = ROOT / "outputs" / "reports"
PROCESSED = ROOT / "data" / "processed"

def read_json(path):
    with open(path, "r", encoding="utf-8") as fh:
        return json.load(fh)

## Dataset and Graph Summary

In [ ]:
summary = read_json(REPORTS / "homepage_summary.json")
pd.DataFrame(summary["dataset"]["split_counts"])

In [ ]:
graph_stats = read_json(REPORTS / "graph_stats_summary.json")
pd.DataFrame([
    {"split": split, "node_count": stats["node_count"], "edge_count": stats["edge_count"], **stats["node_counts_by_type"]}
    for split, stats in graph_stats.items()
])

## Evaluation and Validation

In [ ]:
metrics = read_json(REPORTS / "metrics.json")
validation = read_json(REPORTS / "graph_validation_summary.json")
metrics, validation

## Processed Tables

In [ ]:
theorems = pd.read_parquet(PROCESSED / "train" / "theorems.parquet")
proof_states = pd.read_parquet(PROCESSED / "train" / "proof_states.parquet")
premises = pd.read_parquet(PROCESSED / "train" / "premises.parquet")
proof_techniques = pd.read_parquet(PROCESSED / "train" / "proof_techniques.parquet")

display(theorems.head())
display(proof_states[["id", "theorem_id", "goal_text", "tactic_idx"]].head())
display(premises[["id", "full_name", "file_path"]].head())
display(proof_techniques.head())

## Retrieval API

In [ ]:
from leanrank_kg.retrieve import (
    explain_premise_match,
    get_difficulty_profile,
    get_graph_neighborhood,
    get_proof_technique_labels,
    retrieve_premises,
    retrieve_similar_theorems,
)

proof_state_id = proof_states.iloc[0]["id"]
theorem_id = theorems.iloc[0]["id"]
premise_id = pd.read_parquet(PROCESSED / "train" / "positive_edges.parquet").iloc[0]["premise_id"]

retrieve_premises(proof_state_id, k=5, split="train")

In [ ]:
retrieve_similar_theorems(theorem_id, k=5, split="train")

In [ ]:
explain_premise_match(proof_state_id, premise_id, split="train")

In [ ]:
get_proof_technique_labels(proof_state_id, split="train"), get_difficulty_profile(proof_state_id, split="train")

In [ ]:
neighborhood = get_graph_neighborhood(proof_state_id, depth=1, split="train")
{"node_count": len(neighborhood["nodes"]), "edge_count": len(neighborhood["edges"])}